<a href="https://colab.research.google.com/github/Lucas-O-S/CalculadoraMedia/blob/main/AdrianaTeste" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Código para análise de intervalos estatísticos

Bibliotecas

In [13]:
import math
import pandas as pd



Função para:
1. Contar o número de dados e organizar-los em ordem crescente(do menor para o maior), não cria outra lista, ela altera a própria lista
2. Calcula o número de classes segundo o número de dados, arredonda pra cima.
3. Calcula amplitude total (distância entre o maior número e o menor)
4. Calcula amplitude das classes, arredonda pra cima. É o intervalo que cada classe deve compreender
5. Cria uma lista com k intervalos de h dos dados
6. Conta quantos dados caem em cada classe (fi) e também calcula o ponto médio (Xm) de cada intervalo
7. Calculo da média agrupada
8. Frequencia acumulada que é utilizada para encontrar a mediana, quartis e a distribuição acumulada dos dados
9. Mediana: encontra pela semelhança de triangulos, isolando a mediana para criar uma formula.
10. Variança, desvio padrão e coeficiente de variação

In [22]:
def calcular_estatisticas(dados):
    #1.len() significa comprimento(quantidade de elementos)
    n = len(dados)
    dados = dados.sort_values()#organiza a própria lista em ordem crescente

    #2. Número de classes (regra: sqrt(n))
    k = math.ceil(math.sqrt(n))

    #3. Amplitude total
    H = max(dados) - min(dados)

    #4. Amplitude de classe
    h = math.ceil(H / k)

    #5. Criar classes
    classes = [] #lista vazia que vai armazenar os intervalos de classes
    inicio = min(dados) #pega o menor valor dos dados, este será o começo

    for i in range(k): #repete o processo k vezes(numero de classes)
        fim = inicio + h #somar h para definir o final da classe com amplitude h
        classes.append((inicio, fim)) #adiciona a classe na lista e guarda como uma tupla(intervalo)
        inicio = fim #atualiza o início da proxima classe que começa onde a anterior terminou

    #6. Frequência por classe
    frequencias = [] #lista onde vai guardar os valores de fi
    pontos_medios = [] #lista dos Xm (ponto médio de cada classe)

    for c in classes: #percorre cada classe0
        fi = sum(1 for x in dados if c[0] <= x < c[1]) #soma 1 se o dado está no intervalo. for x in dados, percorrer todos os dados e <= x < inclui o início e exclui o fim
        frequencias.append(fi) #guarda a frequencia daquela classe
        pontos_medios.append((c[0] + (c[1] - 1)) / 2) #calcula o centro da classe, novamente expluindo o limite o fim

    # Ajuste da última classe (incluir valor máximo)
    frequencias[-1] += sum(1 for x in dados if x == max(dados)) #corrige a ultima classe e soma quantas vezes o maior valor aparece. Inclui o fim.

    #7. Média agrupada
    soma = sum(pm * f for pm, f in zip(pontos_medios, frequencias)) #o zip(x, y) junta duas listas x e y, fica (x1, y1), (x2,y2), x3,y3)...
    #soma de Xm*Fi
    media = soma / n

    #8. Frequência acumulada
    fa = [] #lista onde guarda os valores da frequencia acumulada
    acumulado = 0
    for f in frequencias:
        acumulado += f #vai guardando e somando os valores
        fa.append(acumulado)

    #9. Mediana (dados agrupados)
    pos = n / 2
    #encontrar a classe da mediana
    for i, f_ac in enumerate(fa):#percorre a frequencia acumulada
        if f_ac >= pos: #encontra a primeira classe que atinge a posição
            classe_mediana = classes[i]
            F_anterior = fa[i-1] if i > 0 else 0 #F_anterior é a frequencia acumulada anterior
            fi = frequencias[i] #frequencia da classe
            L = classe_mediana[0] #limite inferior da classe
            break

    mediana = L + ((pos - F_anterior) / fi) * h #como a formula da semelhança de triangulos fica quando isolamos a mediana

    # Medidas de dispersão
    # Variância
    variancia = sum(f * (pm - media) ** 2 for pm, f in zip(pontos_medios, frequencias)) / n
    #pm - media calcula a distância até a média
    #**2 eleva ao quadrado para que todos valores fiquem posivos
    #em seguida multiplica pela frequencia f
    #soma tudo e divide por n
    #isso para cada frequencia f presente na lista formada pelo zip de pontos medios e frequencias

    # Desvio padrão: mede o espalhamento dos dados, quanto maior, mais disperso
    desvio = math.sqrt(variancia)

    # Coeficiente de variação: menor que 20% o sistema é estável
    cv = (desvio / media) * 100

    # Cria a Tabela com os intervalos
    tabela = pd.DataFrame({
        "Classe": classes,
        "fi": frequencias,
        "Xm": pontos_medios,
        "fa": fa
    })

    return {
        "tabela": tabela,
        "media": media,
        "mediana": mediana,
        "variancia": variancia,
        "desvio_padrao": desvio,
        "cv": cv
    }

Como os dados vão ser pegos? se for pra digitar no código precisa criar uma lista dados[] e colocar os valores separados por virgula

# Resultado

In [27]:
data = pd.read_excel("Data.xlsx", header=None)

data = pd.Series(data.values.flatten())



resultado = calcular_estatisticas(data)
print("Tabela de Frequência:\n")
print(resultado["tabela"])

print("\nMédia:", resultado["media"])
print("Mediana:", resultado["mediana"])
print("Variância:", resultado["variancia"])
print("Desvio padrão:", resultado["desvio_padrao"])
print("Coeficiente de Variação (%):", resultado["cv"])

9
Tabela de Frequência:

     Classe  fi    Xm  fa
0  (45, 52)   5  48.0   5
1  (52, 59)  10  55.0  15
2  (59, 66)   9  62.0  24
3  (66, 73)   8  69.0  32
4  (73, 80)   4  76.0  36
5  (80, 87)   3  83.0  39
6  (87, 94)   2  90.0  41

Média: 65.825
Mediana: 62.888888888888886
Variância: 136.01764062500004
Desvio padrão: 11.662660100723164
Coeficiente de Variação (%): 17.7176758081628


In [28]:
# Debugging dataset and frequency calculation
print("n (len data):", len(data))
print("data max/min:", data.max(), data.min())
print("counts of max value:", sum(1 for x in data if x == data.max()))

# replicate calculation from function for debugging
import math

n = len(data)
k = math.ceil(math.sqrt(n))
H = max(data) - min(data)
h = math.ceil(H/k)

classes = []
inicio = min(data)
for i in range(k):
    fim = inicio + h
    classes.append((inicio, fim))
    inicio = fim

frequencias = []
for c in classes:
    fi = sum(1 for x in data if c[0] <= x < c[1])
    frequencias.append(fi)

print("classes:", classes)
print("frequencias before adjust:", frequencias, "sum", sum(frequencias))
print("max value:", max(data))
adjust = sum(1 for x in data if x == max(data))
print("adjust count:", ajust)
if ajust:
    frequencias[-1] += ajust
print("frequencias after adjust:", frequencias, "sum", sum(frequencias))

n (len data): 40
data max/min: 88 45
counts of max value: 1
classes: [(45, 52), (52, 59), (59, 66), (66, 73), (73, 80), (80, 87), (87, 94)]
frequencias before adjust: [5, 10, 9, 8, 4, 3, 1] sum 40
max value: 88


NameError: name 'ajust' is not defined